# Runtime curve walkthrough: wildfire × real estate

Reads the proposed artifact and calls the curves through the cell's reference evaluator, the way a Hazard M3
consumer would. Damage only: burn probability is frequency and never enters the damage call. See
[`00_curve_curation_walkthrough.ipynb`](00_curve_curation_walkthrough.ipynb) for how the curves are built.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "docs/cells").is_dir() and (p / "scripts/reference_helpers").is_dir():
            return p
    raise RuntimeError("Run this notebook from inside the Damage_Modeling repository.")


ROOT = find_repo_root()
CELL_ID = "wildfire_real_estate"
PROP = ROOT / "docs/cells" / CELL_ID / "proposed"
ARTIFACT_PATH = PROP / "wildfire_real_estate__model_v0_1__docs_r1__curve_artifact.json"
REG = "wildfire_real_estate__model_v0_1__docs_r1"  # tag of the cell registers (source, claims, tiers, decisions)
OUT = ROOT / "notebooks/wildfire/real_estate/outputs"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts/reference_helpers"))
import wildfire_real_estate_curve_eval as ev  # the cell's reference evaluator

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

ART = json.loads(ARTIFACT_PATH.read_text())
# Replacement cost new from the Hazard reference profiles r1 (Q3 2025 Dallas prices). Illustration only:
# the artifacts withhold scenario dollars until the consumer qualifies each building value.
BUDGETS_USD = {"RE_RES_SFD_1S_SLAB": 405_000, "RE_IND_WAREHOUSE": 11_000_000, "RE_COM_OFFICE_3S_SLAB": 10_800_000}
NAMES = {"RE_RES_SFD_1S_SLAB": "House", "RE_IND_WAREHOUSE": "Warehouse", "RE_COM_OFFICE_3S_SLAB": "Office",
         "RE_COM_LOWRISE": "Retail (unbound)"}


def table(name: str) -> pd.DataFrame:
    return pd.read_csv(PROP / name)


def save(fig, name: str) -> None:
    fig.savefig(OUT / name, dpi=110, bbox_inches="tight", metadata={"Software": None})


def decision_log(path: Path) -> pd.DataFrame:
    """Decision-log rows: ID, whether JL decided it, and the decision text (shortened)."""
    rows = []
    for line in path.read_text().splitlines():
        if not line.startswith("| D-"):
            continue
        cells = [c.strip() for c in line.strip().strip("|").split(" | ")]
        text = re.sub(r"\*\*", "", cells[1])
        rows.append({"id": cells[0], "status": "JL decided" if "JL decided" in cells[1] else "proposed",
                      "decision": text if len(text) <= 240 else text[:240] + " …"})
    return pd.DataFrame(rows)


print("artifact:", ARTIFACT_PATH.relative_to(ROOT))
print(ART["cell_id"], "|", ART["semantic_damage_model_version"], "|", ART["damage_code_id"],
      "| promotion:", ART["promotion_status"], "| canonical:", ART["canonical_runtime_artifact"])


def call(request: dict) -> dict:
    """Send one request through the cell's reference evaluator; return a flat row."""
    try:
        out = ev.evaluate_damage_call(ART, request)
    except ev.DamageCallError as err:
        return {"status": "error", "curve_id": None, "damage_ratio": None, "reason": err.code}
    r = out["failure_unit_results"][0]
    return {"status": r["status"], "curve_id": r["curve_id"], "damage_ratio": r["scalar_central_dr"],
            "reason": "; ".join(r["withheld_reason_codes"]) or None}

PRODUCT = "USFS_RDS_2016_0034_3_270M"  # the FSim product the class states come from
CLASS = {0: "no event", 1: "< 2 ft", 2: "2-4 ft", 3: "4-6 ft", 4: "6-8 ft", 5: "8-12 ft", 6: "12+ ft"}

artifact: docs/cells/wildfire_real_estate/proposed/wildfire_real_estate__model_v0_1__docs_r1__curve_artifact.json
wildfire_real_estate | model v0.1 | WILDFIRE_REAL_ESTATE_WRC_FSIM_SCREENING_V0_1 | promotion: proposed | canonical: False


## 1. What the artifact contains

In [2]:
identity = pd.DataFrame([
    {"field": "cell", "value": ART["cell_id"]},
    {"field": "damage code", "value": ART["damage_code_id"]},
    {"field": "model / docs", "value": f'{ART["semantic_damage_model_version"]} / {ART["documentation_revision"]}'},
    {"field": "lifecycle / promotion", "value": f'{ART["lifecycle_state"]} / {ART["promotion_status"]}'},
    {"field": "canonical runtime artifact", "value": ART["canonical_runtime_artifact"]},
    {"field": "model grade", "value": ART["model_grade"]},
])
display(identity)
units = pd.DataFrame([{k: u.get(k) for k in ("id", "treatment", "denominator")} for u in ART["failure_units"]])
display(units)
pathways = pd.DataFrame([{
    "pathway": p["pathway_id"],
    "axis field": p["hazard_axis"]["preferred_input_field"],
    "unit": p["hazard_axis"]["unit"],
    "valid range": p["hazard_axis"]["valid_range"],
    "selectors": ", ".join(s["field"] for s in p["selector_logic"] if s.get("required")),
    "conditioners": ", ".join(c["field"] for c in p["conditioner_logic"]) or "-",
    "records": len(p["curve_records"]),
} for p in ART["pathways"]])
display(pathways)

,field,value
0,cell,wildfire_real_estate
1,damage code,WILDFIRE_REAL_ESTATE_WRC_FSIM_SCREENING_V0_1
2,model / docs,model v0.1 / docs r1
3,lifecycle / promotion,scaffold / proposed
4,canonical runtime artifact,False
5,model grade,screening_agency_response_function_with_inspection_building_factors


,id,treatment,denominator
0,WFRE_BUILDING_STRUCTURE,primary_nonzero,"structure replacement cost new of the same reference building (no land, contents, site works or depreciation)"
1,WFRE_BUILDING_CONTENTS,withheld,contents value (deferred)


,pathway,axis field,unit,valid range,selectors,conditioners,records
0,wildfire_building_exposure,conditional_flame_length_class_state,source_native_categorical_class,"[0, 6]","reference_building_id, source_wildfire_product_id",-,3


## 2. Request contract

Two required selectors (the building and the FSim product) and an exact integer class state.

In [3]:
display(pd.DataFrame(ART["evaluation_contract"]["accepted_axis_payloads"]))
display(pd.DataFrame(ART["pathways"][0]["selector_logic"])[["field", "required", "allowed", "effect"]])
print(ART["pathways"][0]["hazard_axis"]["extrapolation_policy"])

,pathway_id,mode,required_fields
0,wildfire_building_exposure,exact_fsim_class_state,[conditional_flame_length_class_state]


,field,required,allowed,effect
0,reference_building_id,True,"[RE_RES_SFD_1S_SLAB, RE_IND_WAREHOUSE, RE_COM_OFFICE_3S_SLAB]",Fixed identity of the reference building; picks the building factor.
1,source_wildfire_product_id,True,[USFS_RDS_2016_0034_3_270M],Pins the FSim flame-length class semantics the curve is bound to.


exact integers only; 0 is the no-event control; reject non-integer, non-numeric and out-of-range states


## 3. Calling the curves at every class

In [4]:
rows = []
for b in ("RE_RES_SFD_1S_SLAB", "RE_IND_WAREHOUSE", "RE_COM_OFFICE_3S_SLAB"):
    for state, label in CLASS.items():
        res = call({"pathway_id": "wildfire_building_exposure", "failure_unit_id": "WFRE_BUILDING_STRUCTURE",
                    "reference_building_id": b, "source_wildfire_product_id": PRODUCT,
                    "conditional_flame_length_class_state": state})
        rows.append({"building": NAMES[b], "class": f"{state}: {label}", "damage_ratio": res["damage_ratio"]})
grid = pd.DataFrame(rows).pivot(index="building", columns="class", values="damage_ratio")
display(grid.style.format("{:.1%}"))

class,0: no event,1: < 2 ft,2: 2-4 ft,3: 4-6 ft,4: 6-8 ft,5: 8-12 ft,6: 12+ ft
building,,,,,,,
House,0.0%,20.0%,35.0%,50.0%,65.0%,80.0%,95.0%
Office,0.0%,18.0%,31.5%,45.0%,58.5%,72.0%,85.5%
Warehouse,0.0%,16.0%,28.0%,40.0%,52.0%,64.0%,76.0%


## 4. Fail-closed behaviour

In [5]:
base = {"pathway_id": "wildfire_building_exposure", "failure_unit_id": "WFRE_BUILDING_STRUCTURE",
        "reference_building_id": "RE_RES_SFD_1S_SLAB", "source_wildfire_product_id": PRODUCT,
        "conditional_flame_length_class_state": 3}
cases = {
    "valid request (class 3)": base,
    "fractional class (2.5)": {**base, "conditional_flame_length_class_state": 2.5},
    "class 7 (out of range)": {**base, "conditional_flame_length_class_state": 7},
    "class as a label": {**base, "conditional_flame_length_class_state": "4-6 ft"},
    "burn probability in the damage call": {**base, "burn_probability": 0.01},
    "missing product id": {k: v for k, v in base.items() if k != "source_wildfire_product_id"},
    "contents": {**base, "failure_unit_id": "WFRE_BUILDING_CONTENTS"},
}
display(pd.DataFrame([{"case": k, **call(v)} for k, v in cases.items()]))

,case,status,curve_id,damage_ratio,reason
0,valid request (class 3),conditional,WFRE_RES_WRC_SHRUB,0.5,NaN
1,fractional class (2.5),error,NaN,NaN,INVALID_CLASS_STATE
2,class 7 (out of range),error,NaN,NaN,INVALID_CLASS_STATE
3,class as a label,error,NaN,NaN,INVALID_CLASS_STATE
4,burn probability in the damage call,error,NaN,NaN,FREQUENCY_FIELD_NOT_ALLOWED_IN_DAMAGE_CALL
5,missing product id,error,NaN,NaN,SOURCE_WILDFIRE_PRODUCT_ID_REQUIRED
6,contents,withheld,NaN,NaN,NO_RUNTIME_CURVE_FOR_PATHWAY_UNIT; CONTENTS_DEFERRED_SEPARATE_FAILURE_UNIT


## 5. From damage ratio to dollars (illustration)

The Hazard profile budgets are used only to show `loss = DR × replacement cost new`; the artifact withholds dollars until Hazard qualifies each value.

In [6]:
rows = []
for b, budget in BUDGETS_USD.items():
    res = call({"pathway_id": "wildfire_building_exposure", "failure_unit_id": "WFRE_BUILDING_STRUCTURE",
                "reference_building_id": b, "source_wildfire_product_id": PRODUCT, "conditional_flame_length_class_state": 4})
    rows.append({"building": NAMES[b], "replacement cost new": budget, "DR at class 4 (6-8 ft)": res["damage_ratio"],
                 "loss (illustration)": res["damage_ratio"] * budget})
display(pd.DataFrame(rows).style.format({"replacement cost new": "${:,.0f}", "DR at class 4 (6-8 ft)": "{:.1%}",
                                         "loss (illustration)": "${:,.0f}"}))

,building,replacement cost new,DR at class 4 (6-8 ft),loss (illustration)
0,House,"$405,000",65.0%,"$263,250"
1,Warehouse,"$11,000,000",52.0%,"$5,720,000"
2,Office,"$10,800,000",58.5%,"$6,318,000"


## 6. Capability

In [7]:
cap = ART["capability_declaration"]
rows = []
for pc in cap["pathway_capabilities"]:
    rows.append({"pathway": pc["pathway_id"], "failure-unit DR": pc["failure_unit_scalar_dr"],
                 "scenario loss": pc["scenario_loss_given_value_basis"], "spread": pc["curve_intrinsic_spread"],
                 "withheld units": ", ".join(w["failure_unit_id"] for w in pc.get("withheld_failure_units", []))})
display(pd.DataFrame(rows))
print("Annual and tail metrics:", cap["consumer_annual_metrics"]["status_before_promotion"],
      "(owner:", cap["consumer_annual_metrics"]["computation_owner"] + ")")
print("Promotion gate:", cap["promotion_gate"]["status"])
for item in cap["promotion_gate"]["required_before_canonical_use"]:
    print("  -", item)

,pathway,failure-unit DR,scenario loss,spread,withheld units
0,wildfire_building_exposure,conditional,withheld,not_carried,WFRE_BUILDING_CONTENTS


Annual and tail metrics: withheld_noncanonical_proposal (owner: downstream_consumer)
Promotion gate: blocked
  - owner accepts D-WFRE-01 to D-WFRE-14
  - consumer value qualification of each reference budget (shared with the other real-estate cells)
  - reviewer accepts the agency-response basis, the shrub default and the building factors
  - independent review of the package


## 7. Not in this notebook

- Frequency and yearly loss belong to Hazard; the NRI comparison is `scripts/reference_helpers/wildfire_real_estate_nri_check.py`
  and the CAL FIRE analysis is `wildfire_real_estate_dins_check.py`.
- The Hazard loader probe is `check_wildfire_real_estate_hazard_loader.py`.